In [1]:
import pandas as pd
import pyam
from NFormat import NFormat
from pathlib import Path

<IPython.core.display.Javascript object>

In [2]:
# Set path to working directory
path = Path.cwd()

# Find message-ix directory
while path.name != 'message-ix':
    path = path.parent

## Data input
### The following gathers demand data from various sources into dataframes

#### Reading in data from [Canada's Energy Future](https://apps.cer-rec.gc.ca/ftrppndc/dflt.aspx?GoCTemplateCulture=en-CA)

This data will be used for specific residential and commerical energy demand, as well as transport demand

In [3]:
energy_future_data = NFormat(path / 'data' / 'downscaling' / 'data' / 'can_energy_future.xlsx', 'can_energy_future')

#### Reading in data from [Statistics Canada](https://www150.statcan.gc.ca/t1/tbl1/en/cv.action?pid=2510002901)

This data will be used for industrial feedstock demand

In [4]:
i_feed = NFormat(path / 'data' / 'downscaling' / 'data' / 'statcan_data.csv', 'statcan')

#### Reading in data from [Natural Resources Canada](https://oee.nrcan.gc.ca/corporate/statistics/neud/dpa/menus/trends/comprehensive_tables/list.cfm)

This data will be used for residential and commerical thermal demand before 2020 (might not be needed?)

In [5]:
space_heating = NFormat(path / 'data' / 'downscaling' / 'data' / 'space_heating.xlsx', 'space_heating')

# Downscaling

In [6]:
# Data from previous msg run for downscaling proxys
msg_data = pd.read_excel(path / 'data' / 'MESSAGE_CA__test__v4.xlsx', sheet_name = 'demand')
msg_data = msg_data[msg_data['year'] >= 2020]
msg_data = msg_data[['year', 'node', 'commodity', 'unit', 'value']]
msg_data = pyam.IamDataFrame(msg_data, model = 'MESSAGEix', scenario = 'None', variable = 'commodity', region = 'node')

In [7]:
# Downscale space heating from territories to NWT, Nunavut, Yukon
territories = space_heating.filter(region = 'Territories')
territories.append(energy_future_data.filter(region = ['Northwest Territories', 'Yukon', 'Nunavut']), inplace = True)
new = territories.downscale_region('Space Heating Usage', region = 'Territories', proxy = 'rc_spec')
space_heating.append(new, inplace = True)
space_heating = space_heating.filter(region = energy_future_data.region)

In [8]:
# Thermal downscaling
therm = msg_data.append(energy_future_data)
i_therm = therm.downscale_region('i_therm', region = 'Canada', proxy = 'i_spec')
rc_therm = therm.downscale_region('rc_therm', region = 'Canada', proxy = 'rc_spec')

In [9]:
# Feedstock downscaling
feed = msg_data.append(energy_future_data)
i_feed_proj = feed.downscale_region('i_feed', region = 'Canada', proxy = 'i_spec')

In [10]:
msg_years = list(range(2020, 2061, 5))

# Gather all data together
total = energy_future_data
total.append(i_feed, inplace = True)
total.append(space_heating, inplace = True)
total.append(i_therm, inplace = True)
total.append(rc_therm, inplace = True)
total.append(i_feed_proj, inplace = True)
total.append(msg_data.filter(year = msg_years, variable = ['i_feed', 'i_therm', 'rc_therm']), inplace = True)
total.append(msg_data.filter(year = [2055, 2060], variable = ['i_spec', 'rc_spec', 'transport']), inplace = True)
final = total.filter(year = [msg_years])

In [11]:
final = final.convert_unit('TJ', to = 'GWh').convert_unit('GWh', to = 'GWa', factor = 1/8760)
final = final.convert_unit('PJ', to = 'GWh').convert_unit('GWh', to = 'GWa', factor = 1/8760)

In [12]:
# Match MESSAGE spatial sets naming
final.rename(region = {'British Columbia': 'BritishColumbia',
                       'New Brunswick': 'NewBrunswick',
                       'Newfoundland and Labrador': 'NewfoundlandandLabrador',
                       'Northwest Territories': 'NorthwestTerritories',
                       'Nova Scotia': 'NovaScotia',
                       'Prince Edward Island': 'PrinceEdwardIsland'}, inplace = True)

In [ ]:
# Export to data folder of repo
final.to_excel(path / 'data' / 'canadian_energy_demand.xlsx')

# Below needs to be tested

In [77]:
end_data = final.filter(year = [2050, 2055, 2060])

In [80]:
end_data.filter(region = 'Canada').timeseries()

2050        2055        2060
model     scenario region variable  unit                                   
MESSAGEix None     Canada i_feed    GWa   33.024618   34.279650   35.534683
                          i_spec    GWa   53.424975   22.498991   23.320478
                          i_therm   GWa   39.947588   41.810019   43.672449
                          rc_spec   GWa   44.149860   56.998558   60.071418
                          rc_therm  GWa   52.113446   52.897783   53.683030
                          transport GWa   79.918823  112.944822  116.240001

In [55]:
end_data = end_data.as_pandas()

In [56]:
end_data

,model,scenario,region,variable,unit,year,value,exclude
0,MESSAGEix,None,Alberta,i_feed,GWa,2050,6.828692,False
1,MESSAGEix,None,Alberta,i_spec,GWa,2050,11.047057,False
2,MESSAGEix,None,Alberta,i_therm,GWa,2050,8.260194,False
3,MESSAGEix,None,Alberta,rc_spec,GWa,2050,5.279680,False
4,MESSAGEix,None,Alberta,rc_therm,GWa,2050,5.648244,False
...,...,...,...,...,...,...,...,...
91,MESSAGEix,None,Yukon,i_spec,GWa,2050,0.002220,False
92,MESSAGEix,None,Yukon,i_therm,GWa,2050,0.001660,False
93,MESSAGEix,None,Yukon,rc_spec,GWa,2050,0.052321,False
94,MESSAGEix,None,Yukon,rc_therm,GWa,2050,0.055974,False


In [58]:
end_data = end_data.pivot(index = ['model', 'scenario', 'region', 'variable', 'unit'], columns = 'year', values = 'value')

In [59]:
end_data[2061] = end_data[2055]/end_data[2050]
end_data[2062] = end_data[2060]/end_data[2050]

In [66]:
end_data = pyam.IamDataFrame(end_data)
end_data = end_data.as_pandas()

In [72]:
alb2055 = end_data[end_data['region'] == 'Alberta'].copy()

In [76]:
alb2055['value']*ratios2055['value']

0     7.088202
1     4.652275
2     8.645300
3     6.816198
4     5.733253
5    21.092904
Name: value, dtype: float64

In [70]:
ratios2055 = end_data[end_data['year'] == 2061].reset_index(drop = True)

In [71]:
ratios2055

,model,scenario,region,variable,unit,year,value,exclude
0,MESSAGEix,None,Canada,i_feed,GWa,2061,1.038003,False
1,MESSAGEix,None,Canada,i_spec,GWa,2061,0.421132,False
2,MESSAGEix,None,Canada,i_therm,GWa,2061,1.046622,False
3,MESSAGEix,None,Canada,rc_spec,GWa,2061,1.291025,False
4,MESSAGEix,None,Canada,rc_therm,GWa,2061,1.015051,False
5,MESSAGEix,None,Canada,transport,GWa,2061,1.413244,False


In [31]:
end_data[2055] = end_data[2050]*end_data[2061]

In [33]:
new = pyam.IamDataFrame(end_data)

In [ ]:
here = final.filter(year = [2050, 2055, 2060])

In [ ]:
here.timeseries()

In [ ]:
provinces = here.region
provinces.remove('Canada')
here.filter(region = provinces).region

In [ ]:
ratios = here.filter(region = 'Canada').as_pandas()

In [ ]:
ratios = ratios.pivot(index = ['model', 'scenario', 'region', 'variable', 'unit'], columns = 'year', values = 'value')

In [ ]:
ratios['2055/2050'] = ratios[2055]/ratios[2050]
ratios['2060/2050'] = ratios[2060]/ratios[2050]

In [ ]:
ratios = pyam.IamDataFrame(ratios)

In [ ]:
ratios = ratios.as_pandas()

In [ ]:
ratios['value'] = ratios

In [ ]:
ratios_prov = here.filter(region = provinces).as_pandas()

In [ ]:
ratios2[2055] = ratios2[2050]*ratios2['2055/2050']

In [ ]:
ratios2

In [ ]:
test = pyam.IamDataFrame(ratios2)